# Phase 1 Image Downloader

Downloads images from HuggingFace `timbrooks/instructpix2pix-clip-filtered` using `phase1_dataset.parquet` for the sample mapping.

**Input:** `phase1_dataset.parquet` (upload to this notebook directly or keep locally)
**Output:** `images/original/` + `images/edited/` + copy of parquet

After running, download the output and upload everything (parquet + images/) together as dataset `phase_1` on Kaggle.

In [ ]:
!pip install -q datasets Pillow pandas

In [ ]:
import os
import shutil
import pandas as pd
from PIL import Image
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Paths ──
IS_KAGGLE = os.path.exists("/kaggle/working")
OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "./output"

# Where is your parquet file?
# Option A: Uploaded as Kaggle dataset "phase_1"
# Option B: Uploaded directly to this notebook via "File > Upload"
PARQUET_CANDIDATES = [
    "/kaggle/input/phase_1/phase1_dataset.parquet",                        # dataset named phase_1
    "/kaggle/input/datasets/tusherbhomik/phase-1/phase1_dataset.parquet",  # dataset named phase-1
    "/kaggle/working/phase1_dataset.parquet",                              # uploaded to notebook
    "./output/phase1_dataset.parquet",                                     # local
    "./phase1_dataset.parquet",                                            # local root
]

PARQUET_PATH = None
for p in PARQUET_CANDIDATES:
    if os.path.exists(p):
        PARQUET_PATH = p
        break

if PARQUET_PATH is None:
    # List /kaggle/input to help debug
    if IS_KAGGLE:
        print("Could not find parquet! Contents of /kaggle/input/:")
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError("phase1_dataset.parquet not found in any expected location!")

IMAGE_DIR_ORIG = os.path.join(OUTPUT_DIR, "images", "original")
IMAGE_DIR_EDIT = os.path.join(OUTPUT_DIR, "images", "edited")
os.makedirs(IMAGE_DIR_ORIG, exist_ok=True)
os.makedirs(IMAGE_DIR_EDIT, exist_ok=True)

print(f"IS_KAGGLE:    {IS_KAGGLE}")
print(f"PARQUET_PATH: {PARQUET_PATH}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

In [ ]:
# Load parquet to get dataset_index -> sample_id mapping
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} samples")
print(df[["sample_id", "dataset_index"]].head())

In [ ]:
# Load HuggingFace dataset
print("Loading HuggingFace dataset (this may take a few minutes)...")
hf_dataset = load_dataset("timbrooks/instructpix2pix-clip-filtered", split="train")
print(f"HF dataset size: {len(hf_dataset)}")

In [ ]:
# Extract and save images for our subset
saved = 0
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Saving images"):
    sample_id = row["sample_id"]
    ds_idx = int(row["dataset_index"])
    
    orig_path = os.path.join(IMAGE_DIR_ORIG, f"{sample_id}.jpg")
    edit_path = os.path.join(IMAGE_DIR_EDIT, f"{sample_id}.jpg")
    
    # Skip if already saved
    if os.path.exists(orig_path) and os.path.exists(edit_path):
        skipped += 1
        continue
    
    sample = hf_dataset[ds_idx]
    
    orig_img = sample["original_image"]
    if isinstance(orig_img, Image.Image):
        orig_img.convert("RGB").save(orig_path, quality=95)
    
    edit_img = sample["edited_image"]
    if isinstance(edit_img, Image.Image):
        edit_img.convert("RGB").save(edit_path, quality=95)
    
    saved += 1

print(f"\nDone! Saved: {saved}, Skipped (already existed): {skipped}")
print(f"Original images: {len(os.listdir(IMAGE_DIR_ORIG))}")
print(f"Edited images: {len(os.listdir(IMAGE_DIR_EDIT))}")

In [ ]:
# Copy parquet to output so everything is in one place
output_parquet = os.path.join(OUTPUT_DIR, "phase1_dataset.parquet")
if not os.path.exists(output_parquet):
    shutil.copy2(PARQUET_PATH, output_parquet)
    print(f"Copied parquet to {output_parquet}")
else:
    print(f"Parquet already at {output_parquet}")

# Verify
print(f"\nOutput contents:")
!ls -la {OUTPUT_DIR}/phase1_dataset.parquet
!echo "Original images:" && ls {OUTPUT_DIR}/images/original/ | head -5
!echo "Edited images:" && ls {OUTPUT_DIR}/images/edited/ | head -5
!echo "" && du -sh {OUTPUT_DIR}/images/

print(f"\nTotal files: parquet + {len(os.listdir(IMAGE_DIR_ORIG))} original + {len(os.listdir(IMAGE_DIR_EDIT))} edited")

## Next Steps

1. **Save this notebook** -> "Save Version" -> "Quick Save"
2. Go to notebook **Output** tab -> click **"New Dataset"** -> name it **`phase_1`**
   - This creates a dataset containing: `phase1_dataset.parquet` + `images/original/` + `images/edited/`
3. In **Phase 2 notebook**, add this single dataset `phase_1` as input
   - The mount path will be something like `/kaggle/input/phase_1/`
   - Parquet: `/kaggle/input/phase_1/phase1_dataset.parquet`
   - Images: `/kaggle/input/phase_1/images/original/000000.jpg`